# Comprensi?n EDA - CustomerChurnX

An?lisis exploratorio del dataset de churn para definir reglas de validaci?n, transformaciones y se?ales ?tiles para el modelado supervisado.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

candidates = [
    Path.cwd() / "Base_de_datos.csv",
    Path.cwd().parent / "Base_de_datos.csv",
    Path.cwd() / "mlops_pipeline" / "Base_de_datos.csv",
]
DATA_PATH = next((path for path in candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No se encontro Base_de_datos.csv desde el directorio actual")
ROOT = DATA_PATH.parent
DATA_PATH

sns.set_theme(style="whitegrid")
df = pd.read_csv(DATA_PATH)
df.head()

## Exploraci?n inicial

In [ ]:
display(df.describe(include="all").T)
print("Shape:", df.shape)
print("Target churn rate:", round(df["churn"].mean(), 4))

In [ ]:
pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_pct": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique()
})

## Caracterizaci?n de variables

- Num?ricas continuas: `avg_session_min`, `notif_click_rate`, `discount_pct_3m`.
- Num?ricas discretas: `age`, `tenure_months`, `signup_month`, `sessions_week`, `support_tickets_3m`, `late_payments_6m`.
- Categ?ricas nominales: `region`, `channel`, `plan`.
- Binarias: `auto_renew`, `churn`.
- Identificador no predictivo: `customer_id`.

## An?lisis univariable

In [ ]:
numeric_cols = ["age", "tenure_months", "sessions_week", "avg_session_min", "notif_click_rate", "support_tickets_3m", "discount_pct_3m", "late_payments_6m"]
df[numeric_cols].hist(figsize=(14, 10), bins=30)
plt.tight_layout()

In [ ]:
categorical_cols = ["region", "channel", "plan", "auto_renew", "churn"]
fig, axes = plt.subplots(1, len(categorical_cols), figsize=(18, 4))
for ax, col in zip(axes, categorical_cols):
    sns.countplot(data=df, x=col, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()

## An?lisis bivariable respecto al objetivo

In [ ]:
churn_by_cat = {}
for col in ["region", "channel", "plan", "auto_renew"]:
    churn_by_cat[col] = df.groupby(col)["churn"].agg(["count", "mean"]).sort_values("mean", ascending=False)
    display(churn_by_cat[col])

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.ravel(), numeric_cols):
    sns.boxplot(data=df, x="churn", y=col, ax=ax)
    ax.set_title(f"{col} vs churn")
plt.tight_layout()

## An?lisis multivariable

In [ ]:
corr = df[numeric_cols + ["auto_renew", "churn"]].corr(numeric_only=True)
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlaci?n")

In [ ]:
pivot = pd.pivot_table(df, values="churn", index="plan", columns="channel", aggfunc="mean")
pivot

## Hallazgos y decisiones para feature engineering

- `customer_id` se elimina del set predictivo porque es identificador.
- Las variables categ?ricas deben codificarse con One-Hot Encoding.
- Conviene crear se?ales de negocio como minutos de uso semanal, tickets por antig?edad, bandera de mora, descuento alto y baja interacci?n.
- La m?trica principal debe combinar ROC-AUC y F1 porque el objetivo es priorizar clientes con riesgo de churn sin ignorar falsos positivos.
- Para monitoreo, se comparar?n ventanas hist?ricas y recientes por `signup_month` para detectar drift.